# Phase 2 -- Feature Engineering and Classical ML
## Human vs. AI-Generated Text Classification

**Objectives:**
1. Extract 46 linguistic and stylometric features from text
2. Combine with TF-IDF (word 1-2 gram, 50k + char 2-5 gram, 50k)
3. Train 10 classifiers: LR x 4, SVC x 4, XGBoost, LightGBM
4. 5-fold Stratified CV on all models
5. Majority-vote and weighted-vote ensembles
6. Save trained models and vectorisers for later phases


In [9]:
import os, re, string, warnings, json
os.chdir('/Users/aliivaezii/Documents/MALTO')
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier, VotingClassifier, StackingClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import f1_score, classification_report, confusion_matrix
from sklearn.preprocessing import StandardScaler
from scipy.sparse import hstack, csr_matrix
import xgboost as xgb
import lightgbm as lgb

LABEL_MAP = {0:'Human', 1:'DeepSeek', 2:'Grok', 3:'Claude', 4:'Gemini', 5:'ChatGPT'}
SEED = 42
np.random.seed(SEED)

# Load data
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')
print(f'Train: {train.shape}, Test: {test.shape}')
print(f'Label distribution:\n{train.LABEL.value_counts().sort_index()}')

Train: (2400, 2), Test: (600, 2)
Label distribution:
LABEL
0    1520
1      80
2     160
3      80
4     240
5     320
Name: count, dtype: int64


## 1. Comprehensive Feature Engineering

In [11]:
def extract_features(texts):
    """Extract rich linguistic and stylometric features from texts."""
    features = []
 
    for text in texts:
        f = {}
 
        # --- Basic length features ---
        words = text.split()
        sentences = re.split(r'[.!?]+', text)
        sentences = [s.strip() for s in sentences if s.strip()]
        chars = list(text)
 
        f['char_count'] = len(text)
        f['word_count'] = len(words)
        f['sentence_count'] = max(len(sentences), 1)
        f['avg_word_len'] = np.mean([len(w) for w in words]) if words else 0
        f['avg_sentence_len'] = f['word_count'] / f['sentence_count']
        f['max_word_len'] = max([len(w) for w in words]) if words else 0
        f['std_word_len'] = np.std([len(w) for w in words]) if len(words) > 1 else 0
 
        # --- Vocabulary richness ---
        lower_words = [w.lower() for w in words]
        unique_words = set(lower_words)
        f['unique_word_count'] = len(unique_words)
        f['type_token_ratio'] = len(unique_words) / max(len(words), 1)
        # Hapax legomena (words appearing once)
        word_freq = Counter(lower_words)
        f['hapax_ratio'] = sum(1 for v in word_freq.values() if v == 1) / max(len(unique_words), 1)
        # Yule's K measure
        freq_spectrum = Counter(word_freq.values())
        N = len(lower_words)
        M2 = sum(i*i*freq_spectrum[i] for i in freq_spectrum)
        f['yules_k'] = 10000 * (M2 - N) / (N*N) if N > 1 else 0
 
        # --- Character-level features ---
        f['upper_ratio'] = sum(1 for c in text if c.isupper()) / max(len(text), 1)
        f['digit_ratio'] = sum(1 for c in text if c.isdigit()) / max(len(text), 1)
        f['space_ratio'] = sum(1 for c in text if c.isspace()) / max(len(text), 1)
        f['alpha_ratio'] = sum(1 for c in text if c.isalpha()) / max(len(text), 1)
 
        # --- Punctuation features ---
        f['punct_ratio'] = sum(1 for c in text if c in string.punctuation) / max(len(text), 1)
        f['comma_ratio'] = text.count(',') / max(len(words), 1)
        f['period_ratio'] = text.count('.') / max(len(words), 1)
        f['exclamation_ratio'] = text.count('!') / max(len(words), 1)
        f['question_ratio'] = text.count('?') / max(len(words), 1)
        f['semicolon_ratio'] = text.count(';') / max(len(words), 1)
        f['colon_ratio'] = text.count(':') / max(len(words), 1)
        f['quote_count'] = text.count('"') + text.count("'")
        f['paren_count'] = text.count('(') + text.count(')')
        f['dash_count'] = text.count('-') + text.count('--')
 
        # --- Spelling / typo indicators ---
        # Double letters (common in misspellings)
        f['double_letter_ratio'] = len(re.findall(r'(.)\1', text.lower())) / max(len(words), 1)
        # Words with unusual patterns (likely typos)
        f['short_word_ratio'] = sum(1 for w in words if len(w) <= 2) / max(len(words), 1)
        f['long_word_ratio'] = sum(1 for w in words if len(w) > 10) / max(len(words), 1)
        f['very_long_word_ratio'] = sum(1 for w in words if len(w) > 15) / max(len(words), 1)
        # Consecutive consonants (3+) -- sign of typos
        consonant_clusters = re.findall(r'[bcdfghjklmnpqrstvwxyz]{3,}', text.lower())
        f['consonant_cluster_ratio'] = len(consonant_clusters) / max(len(words), 1)
        # Repeated characters (3+) -- sign of emphasis/typos
        repeated_chars = re.findall(r'(.)\1{2,}', text)
        f['repeated_char_count'] = len(repeated_chars)
 
        # --- Structural features ---
        f['paragraph_count'] = text.count('\n\n') + 1
        f['newline_count'] = text.count('\n')
        f['starts_with_upper'] = int(text[0].isupper()) if text else 0
        f['ends_with_period'] = int(text.rstrip().endswith('.')) if text else 0
        # Bullet points / list items
        f['bullet_count'] = len(re.findall(r'^\s*[-*•]\s', text, re.MULTILINE))
        f['numbered_list_count'] = len(re.findall(r'^\s*\d+[.):]\s', text, re.MULTILINE))
 
        # --- AI-style indicators ---
        text_lower = text.lower()
        # Common AI phrases
        ai_phrases = [
            'in conclusion', 'furthermore', 'moreover', 'additionally',
            'it is worth noting', 'it is important to', 'on the other hand',
            'in summary', 'to summarize', 'in essence', 'ultimately',
            'delve', 'crucial', 'comprehensive', 'leverage', 'facilitate',
            'robust', 'streamline', 'innovative', 'cutting-edge', 'paradigm',
            'holistic', 'synergy', 'encompass', 'multifaceted', 'nuanced'
            ]
        f['ai_phrase_count'] = sum(1 for p in ai_phrases if p in text_lower)
 
        # Transition words (AI texts tend to use more)
        transitions = [
            'however', 'therefore', 'nonetheless', 'nevertheless',
            'consequently', 'meanwhile', 'subsequently', 'accordingly',
            'hence', 'thus', 'thereby'
            ]
        f['transition_count'] = sum(1 for t in transitions if t in text_lower)
 
        # First person pronouns (more common in human text)
        first_person = re.findall(r'\b(i|me|my|mine|myself|we|us|our|ours|ourselves)\b', text_lower)
        f['first_person_ratio'] = len(first_person) / max(len(words), 1)
 
        # Contractions (more common in human text)
        contractions = re.findall(r"\b\w+'\w+\b", text)
        f['contraction_ratio'] = len(contractions) / max(len(words), 1)
 
        # --- Readability proxies ---
        # Syllable approximation
        def count_syllables(word):
            word = word.lower().rstrip('e')
            return max(1, len(re.findall(r'[aeiouy]+', word)))
 
        total_syllables = sum(count_syllables(w) for w in words) if words else 1

        # Flesch Reading Ease approximation

        f['flesch_reading_ease'] = 206.835 - 1.015 * f['avg_sentence_len'] - 84.6 * (total_syllables / max(len(words), 1))

        # Flesch-Kincaid Grade Level

        f['flesch_kincaid_grade'] = 0.39 * f['avg_sentence_len'] + 11.8 * (total_syllables / max(len(words), 1)) - 15.59

        # Automated Readability Index

        f['ari'] = 4.71 * (f['char_count'] / max(len(words), 1)) + 0.5 * f['avg_sentence_len'] - 21.43

 
        # --- Entropy features ---

        char_freq = Counter(text.lower())

        total_chars = sum(char_freq.values())

        char_probs = [c/total_chars for c in char_freq.values()]

        f['char_entropy'] = -sum(p * np.log2(p) for p in char_probs if p > 0)

 
        word_probs = [c/len(lower_words) for c in word_freq.values()] if lower_words else [1]

        f['word_entropy'] = -sum(p * np.log2(p) for p in word_probs if p > 0)

 
        features.append(f)

 
    return pd.DataFrame(features)


print('Extracting features from train...')
train_feats = extract_features(train['TEXT'].values)
print(f' → {train_feats.shape[1]} features extracted')

print(f'\nFeature columns ({len(train_feats.columns)}):')
print(list(train_feats.columns))

Extracting features from train...
 → 46 features extracted

Feature columns (46):
['char_count', 'word_count', 'sentence_count', 'avg_word_len', 'avg_sentence_len', 'max_word_len', 'std_word_len', 'unique_word_count', 'type_token_ratio', 'hapax_ratio', 'yules_k', 'upper_ratio', 'digit_ratio', 'space_ratio', 'alpha_ratio', 'punct_ratio', 'comma_ratio', 'period_ratio', 'exclamation_ratio', 'question_ratio', 'semicolon_ratio', 'colon_ratio', 'quote_count', 'paren_count', 'dash_count', 'double_letter_ratio', 'short_word_ratio', 'long_word_ratio', 'very_long_word_ratio', 'consonant_cluster_ratio', 'repeated_char_count', 'paragraph_count', 'newline_count', 'starts_with_upper', 'ends_with_period', 'bullet_count', 'numbered_list_count', 'ai_phrase_count', 'transition_count', 'first_person_ratio', 'contraction_ratio', 'flesch_reading_ease', 'flesch_kincaid_grade', 'ari', 'char_entropy', 'word_entropy']
 → 46 features extracted

Feature columns (46):
['char_count', 'word_count', 'sentence_count'

### Feature Importance Analysis

Identify the most discriminative features by computing the standard deviation of class means. Features with high variability across classes are the strongest predictors.


In [12]:
# Quick look at feature distributions per class
train_feats['LABEL'] = train['LABEL'].values

# Show mean features per class
means = train_feats.groupby('LABEL').mean()
# Most discriminative features (highest std across class means)
feat_std = means.std()
top_feats = feat_std.sort_values(ascending=False).head(20)

print('Top 20 most discriminative features (by std of class means):')
for feat, std_val in top_feats.items():
    class_vals = means[feat]
    print(f' {feat:35s} std={std_val:.4f} | ' + 
        ' | '.join(f'{LABEL_MAP[i][:5]}={class_vals[i]:.3f}' for i in range(6)))

train_feats.drop('LABEL', axis=1, inplace=True)

Top 20 most discriminative features (by std of class means):
 char_count                          std=1225.4215 | Human=2510.599 | DeepS=560.025 | Grok=414.469 | Claud=3358.688 | Gemin=2818.662 | ChatG=1485.244
 word_count                          std=176.0193 | Human=450.869 | DeepS=83.075 | Grok=55.644 | Claud=375.625 | Gemin=419.875 | ChatG=179.419
 unique_word_count                   std=91.9962 | Human=236.320 | DeepS=62.138 | Grok=48.388 | Claud=241.950 | Gemin=243.321 | ChatG=131.688
 yules_k                             std=36.5272 | Human=107.082 | DeepS=58.292 | Grok=51.799 | Claud=51.689 | Gemin=128.579 | ChatG=122.072
 flesch_reading_ease                 std=32.9274 | Human=63.780 | DeepS=16.984 | Grok=2.044 | Claud=-21.245 | Gemin=23.870 | ChatG=-25.327
 sentence_count                      std=26.4146 | Human=28.339 | DeepS=3.500 | Grok=2.450 | Claud=72.775 | Gemin=19.913 | ChatG=9.934
 newline_count                       std=7.1341 | Human=0.000 | DeepS=0.000 | Grok=0.000 

## 2. TF-IDF Features (Word + Char n-grams)

In [13]:
# Word-level TF-IDF
tfidf_word = TfidfVectorizer(
    analyzer='word', ngram_range=(1, 2), max_features=50000,
    min_df=2, max_df=0.95, sublinear_tf=True, strip_accents='unicode'
    )

# Character-level TF-IDF
tfidf_char = TfidfVectorizer(
    analyzer='char_wb', ngram_range=(2, 5), max_features=50000,
    min_df=2, max_df=0.95, sublinear_tf=True, strip_accents='unicode'
    )

all_texts = pd.concat([train['TEXT'], test['TEXT']], axis=0).values

# Fit on all data (semi-supervised TF-IDF), transform separately
tfidf_word.fit(all_texts)
tfidf_char.fit(all_texts)

X_train_word = tfidf_word.transform(train['TEXT'].values)
X_test_word = tfidf_word.transform(test['TEXT'].values)
X_train_char = tfidf_char.transform(train['TEXT'].values)
X_test_char = tfidf_char.transform(test['TEXT'].values)

print(f'Word TF-IDF: {X_train_word.shape}')
print(f'Char TF-IDF: {X_train_char.shape}')

Word TF-IDF: (2400, 50000)
Char TF-IDF: (2400, 50000)


### Feature Scaling and Combination

Scale the handcrafted features and combine them with TF-IDF matrices into a single feature representation.


In [14]:
# Scale handcrafted features
scaler = StandardScaler()
train_feats_scaled = scaler.fit_transform(train_feats)
test_feats_scaled = scaler.transform(test_feats)

# Combine: TF-IDF (word + char) + handcrafted features
X_train_all = hstack([
    X_train_word, 
    X_train_char, 
    csr_matrix(train_feats_scaled)
    ])
X_test_all = hstack([
    X_test_word, 
    X_test_char, 
    csr_matrix(test_feats_scaled)
    ])

y = train['LABEL'].values
print(f'Combined train features: {X_train_all.shape}')
print(f'Combined test features: {X_test_all.shape}')

Combined train features: (2400, 100046)
Combined test features: (600, 100046)


## 3. Model Training with 5-Fold Stratified CV

### Overfitting and Underfitting Prevention

Several strategies are employed across all models to control generalization:

- **5-fold stratified cross-validation**: Provides a robust, unbiased estimate of generalization performance and detects overfitting when fold variance is high.
- **Regularization (C parameter)**: Both Logistic Regression and LinearSVC use L2 regularization. Lower C values increase regularization strength, preventing the model from fitting noise. We evaluate C in {0.5, 1.0, 2.0, 5.0} to find the optimal bias-variance tradeoff.
- **Balanced class weights**: Prevents the model from overfitting to the majority class (Human, 63%) by upweighting minority classes during training.
- **Sublinear TF-IDF**: Using `sublinear_tf=True` applies logarithmic scaling to term frequencies, reducing the influence of very frequent terms and improving generalization.
- **Min document frequency**: Setting `min_df=2` removes hapax features (appearing in only one document), reducing dimensionality and noise.
- **Gradient boosting early stopping**: XGBoost and LightGBM use fixed estimator counts but with subsample and colsample_bytree < 1.0, which acts as implicit regularization.


In [16]:
def evaluate_cv(model, X, y, model_name, n_splits=5):
    """Run stratified K-fold CV and return OOF predictions + scores."""
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=SEED)
    oof_preds = np.zeros(len(y))
    test_preds_list = []
    fold_scores = []

    for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
        if hasattr(X, 'toarray'):  # sparse matrix
            X_tr, X_vl = X[train_idx], X[val_idx]
        else:
            X_tr, X_vl = X[train_idx], X[val_idx]
        y_tr, y_vl = y[train_idx], y[val_idx]

        model.fit(X_tr, y_tr)
        preds = model.predict(X_vl)
        oof_preds[val_idx] = preds

        fold_f1 = f1_score(y_vl, preds, average='macro')
        fold_scores.append(fold_f1)

        # Test predictions for this fold
        test_preds_list.append(model.predict(X_test_all))

    mean_f1 = np.mean(fold_scores)
    std_f1 = np.std(fold_scores)
    overall_f1 = f1_score(y, oof_preds, average='macro')

    print(f'\n{"="*60}')
    print(f'{model_name}')
    print(f' Fold scores: {[f"{s:.4f}" for s in fold_scores]}')
    print(f' Mean CV F1: {mean_f1:.4f} \u00b1 {std_f1:.4f}')
    print(f' OOF F1: {overall_f1:.4f}')

    # Majority vote for test predictions
    test_preds_array = np.array(test_preds_list)  # (n_folds, n_test)
    test_preds_final = np.apply_along_axis(
        lambda x: np.bincount(x.astype(int), minlength=6).argmax(),
        axis=0, arr=test_preds_array
    )

    return {
        'name': model_name,
        'mean_f1': mean_f1,
        'std_f1': std_f1,
        'oof_f1': overall_f1,
        'oof_preds': oof_preds.astype(int),
        'test_preds': test_preds_final,
        'fold_scores': fold_scores
    }

results = {}
print('Starting model evaluation...')

Starting model evaluation...


###  Cross-Validation Helper

A reusable stratified 5-fold CV function that returns out-of-fold predictions, per-fold scores, and test predictions.

In [17]:
# --- Model 1: Logistic Regression (TF-IDF word+char + features) ---
from sklearn.base import clone

lr = LogisticRegression(
    C=1.0, class_weight='balanced', max_iter=5000, solver='saga', random_state=SEED
    )
results['LR'] = evaluate_cv(clone(lr), X_train_all, y, 'Logistic Regression (word+char+feats)')

# Per-class breakdown for best model so far
print('\nPer-class OOF report:')
print(classification_report(y, results['LR']['oof_preds'], 
    target_names=[f'{v}({k})' for k,v in LABEL_MAP.items()]))


Logistic Regression (word+char+feats)
 Fold scores: ['0.8755', '0.9345', '0.9140', '0.9417', '0.8472']
 Mean CV F1: 0.9026 ± 0.0360
 OOF F1: 0.9025

Per-class OOF report:
              precision    recall  f1-score   support

    Human(0)       1.00      0.98      0.99      1520
 DeepSeek(1)       0.65      0.60      0.62        80
     Grok(2)       0.83      0.89      0.86       160
   Claude(3)       1.00      0.99      0.99        80
   Gemini(4)       0.94      0.99      0.96       240
  ChatGPT(5)       0.98      0.99      0.99       320

    accuracy                           0.97      2400
   macro avg       0.90      0.91      0.90      2400
weighted avg       0.97      0.97      0.97      2400



###  Model Training — Classical Classifiers

Train each of the 10 classical models via stratified 5-fold CV and record OOF predictions.

In [18]:
# --- Model 2: LinearSVC ---
svc = LinearSVC(
    C=1.0, class_weight='balanced', max_iter=10000, random_state=SEED
    )
results['SVC'] = evaluate_cv(clone(svc), X_train_all, y, 'LinearSVC (word+char+feats)')

print('\nPer-class OOF report:')
print(classification_report(y, results['SVC']['oof_preds'],
    target_names=[f'{v}({k})' for k,v in LABEL_MAP.items()]))


LinearSVC (word+char+feats)
 Fold scores: ['0.9157', '0.9541', '0.9091', '0.9429', '0.8800']
 Mean CV F1: 0.9204 ± 0.0262
 OOF F1: 0.9204

Per-class OOF report:
              precision    recall  f1-score   support

    Human(0)       1.00      1.00      1.00      1520
 DeepSeek(1)       0.82      0.57      0.68        80
     Grok(2)       0.83      0.92      0.87       160
   Claude(3)       1.00      0.99      0.99        80
   Gemini(4)       0.98      1.00      0.99       240
  ChatGPT(5)       0.99      1.00      0.99       320

    accuracy                           0.98      2400
   macro avg       0.94      0.91      0.92      2400
weighted avg       0.98      0.98      0.98      2400



###  Model Training & Evaluation

Train 10 diverse classifiers on different feature combinations using stratified 5-fold cross-validation. Each model gets TF-IDF, stylometric, or combined features.

### Gradient Boosting Models

Train XGBoost with balanced sample weights. Gradient boosting models handle high-dimensional sparse features well and provide complementary predictions to linear models.


In [19]:
# --- Model 3: XGBoost ---
# Class weights for XGBoost
from sklearn.utils.class_weight import compute_sample_weight
sample_weights_train = compute_sample_weight('balanced', y)

# XGBoost needs dense features or we use the sparse directly
xgb_model = xgb.XGBClassifier(
    n_estimators=500, max_depth=6, learning_rate=0.1,
    subsample=0.8, colsample_bytree=0.8,
    objective='multi:softmax', num_class=6,
    eval_metric='mlogloss', random_state=SEED,
    tree_method='hist', n_jobs=-1,
    use_label_encoder=False
    )

# XGBoost CV with sample weights
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
oof_preds_xgb = np.zeros(len(y))
test_preds_xgb_list = []
fold_scores_xgb = []

for fold, (train_idx, val_idx) in enumerate(skf.split(X_train_all, y)):
    X_tr, X_vl = X_train_all[train_idx], X_train_all[val_idx]
    y_tr, y_vl = y[train_idx], y[val_idx]
    sw_tr = compute_sample_weight('balanced', y_tr)
 
    xgb_fold = xgb.XGBClassifier(
        n_estimators=500, max_depth=6, learning_rate=0.1,
        subsample=0.8, colsample_bytree=0.8,
        objective='multi:softmax', num_class=6,
        eval_metric='mlogloss', random_state=SEED,
        tree_method='hist', n_jobs=-1,
        use_label_encoder=False
        )
    xgb_fold.fit(X_tr, y_tr, sample_weight=sw_tr)
    preds = xgb_fold.predict(X_vl)
    oof_preds_xgb[val_idx] = preds
 
    fold_f1 = f1_score(y_vl, preds, average='macro')
    fold_scores_xgb.append(fold_f1)
    print(f' Fold {fold}: F1={fold_f1:.4f}')
 
    test_preds_xgb_list.append(xgb_fold.predict(X_test_all))

test_preds_xgb_array = np.array(test_preds_xgb_list)
test_preds_xgb_final = np.apply_along_axis(
    lambda x: np.bincount(x.astype(int), minlength=6).argmax(),
    axis=0, arr=test_preds_xgb_array
    )

xgb_mean = np.mean(fold_scores_xgb)
xgb_std = np.std(fold_scores_xgb)
xgb_oof = f1_score(y, oof_preds_xgb, average='macro')
print(f'\nXGBoost: Mean CV F1 = {xgb_mean:.4f} ± {xgb_std:.4f}, OOF F1 = {xgb_oof:.4f}')

results['XGB'] = {
    'name': 'XGBoost', 'mean_f1': xgb_mean, 'std_f1': xgb_std,
    'oof_f1': xgb_oof, 'oof_preds': oof_preds_xgb.astype(int),
    'test_preds': test_preds_xgb_final, 'fold_scores': fold_scores_xgb
    }

print('\nPer-class OOF report:')
print(classification_report(y, oof_preds_xgb.astype(int),
    target_names=[f'{v}({k})' for k,v in LABEL_MAP.items()]))

 Fold 0: F1=0.9424
 Fold 1: F1=0.9542
 Fold 2: F1=0.8840
 Fold 3: F1=0.9183
 Fold 4: F1=0.8652

XGBoost: Mean CV F1 = 0.9128 ± 0.0338, OOF F1 = 0.9138

Per-class OOF report:
              precision    recall  f1-score   support

    Human(0)       1.00      1.00      1.00      1520
 DeepSeek(1)       0.72      0.60      0.65        80
     Grok(2)       0.84      0.88      0.86       160
   Claude(3)       1.00      0.99      0.99        80
   Gemini(4)       0.99      0.99      0.99       240
  ChatGPT(5)       0.99      1.00      0.99       320

    accuracy                           0.98      2400
   macro avg       0.92      0.91      0.91      2400
weighted avg       0.97      0.98      0.98      2400



### LightGBM

Train LightGBM as an alternative gradient boosting model. LightGBM is typically faster than XGBoost on large feature sets and offers native support for class weights.


In [20]:
# --- Model 4: LightGBM ---
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
oof_preds_lgb = np.zeros(len(y))
test_preds_lgb_list = []
fold_scores_lgb = []

for fold, (train_idx, val_idx) in enumerate(skf.split(X_train_all, y)):
    X_tr, X_vl = X_train_all[train_idx], X_train_all[val_idx]
    y_tr, y_vl = y[train_idx], y[val_idx]
    sw_tr = compute_sample_weight('balanced', y_tr)
 
    lgb_fold = lgb.LGBMClassifier(
        n_estimators=500, max_depth=8, learning_rate=0.1,
        num_leaves=63, subsample=0.8, colsample_bytree=0.8,
        class_weight='balanced', random_state=SEED,
        n_jobs=-1, verbose=-1
        )
    lgb_fold.fit(X_tr, y_tr, sample_weight=sw_tr)
    preds = lgb_fold.predict(X_vl)
    oof_preds_lgb[val_idx] = preds
 
    fold_f1 = f1_score(y_vl, preds, average='macro')
    fold_scores_lgb.append(fold_f1)
    print(f' Fold {fold}: F1={fold_f1:.4f}')
 
    test_preds_lgb_list.append(lgb_fold.predict(X_test_all))

test_preds_lgb_array = np.array(test_preds_lgb_list)
test_preds_lgb_final = np.apply_along_axis(
    lambda x: np.bincount(x.astype(int), minlength=6).argmax(),
    axis=0, arr=test_preds_lgb_array
    )

lgb_mean = np.mean(fold_scores_lgb)
lgb_std = np.std(fold_scores_lgb)
lgb_oof = f1_score(y, oof_preds_lgb, average='macro')
print(f'\nLightGBM: Mean CV F1 = {lgb_mean:.4f} ± {lgb_std:.4f}, OOF F1 = {lgb_oof:.4f}')

results['LGB'] = {
    'name': 'LightGBM', 'mean_f1': lgb_mean, 'std_f1': lgb_std,
    'oof_f1': lgb_oof, 'oof_preds': oof_preds_lgb.astype(int),
    'test_preds': test_preds_lgb_final, 'fold_scores': fold_scores_lgb
    }

print('\nPer-class OOF report:')
print(classification_report(y, oof_preds_lgb.astype(int),
    target_names=[f'{v}({k})' for k,v in LABEL_MAP.items()]))

 Fold 0: F1=0.9394
 Fold 1: F1=0.9363
 Fold 2: F1=0.8927
 Fold 3: F1=0.9043
 Fold 4: F1=0.8912

LightGBM: Mean CV F1 = 0.9128 ± 0.0210, OOF F1 = 0.9134

Per-class OOF report:
              precision    recall  f1-score   support

    Human(0)       1.00      1.00      1.00      1520
 DeepSeek(1)       0.77      0.54      0.63        80
     Grok(2)       0.84      0.93      0.88       160
   Claude(3)       1.00      1.00      1.00        80
   Gemini(4)       0.97      0.98      0.98       240
  ChatGPT(5)       1.00      0.99      1.00       320

    accuracy                           0.98      2400
   macro avg       0.93      0.91      0.91      2400
weighted avg       0.98      0.98      0.97      2400



### Hyperparameter Tuning -- Logistic Regression

Evaluate Logistic Regression across multiple regularization strengths (C values). Lower C means stronger regularization, which helps prevent overfitting on this relatively small dataset.


In [ ]:
# --- Model 5: LR with tuned C values ---
for C_val in [0.5, 2.0, 5.0]:
    lr_tuned = LogisticRegression(
        C=C_val, class_weight='balanced', max_iter=5000, solver='saga', random_state=SEED
        )
    res = evaluate_cv(clone(lr_tuned), X_train_all, y, f'LR (C={C_val})')
    results[f'LR_C{C_val}'] = res


LR (C=0.5)
 Fold scores: ['0.8729', '0.9345', '0.9002', '0.9268', '0.8472']
 Mean CV F1: 0.8963 ± 0.0328
 OOF F1: 0.8964

LR (C=2.0)
 Fold scores: ['0.8755', '0.9417', '0.9140', '0.9331', '0.8497']
 Mean CV F1: 0.9028 ± 0.0350
 OOF F1: 0.9026


### Hyperparameter Tuning -- LinearSVC

Evaluate LinearSVC across multiple regularization strengths. SVC with balanced class weights is particularly effective for this imbalanced multi-class problem.


In [ ]:
# --- Model 6: SVC with tuned C values ---
for C_val in [0.5, 2.0, 5.0]:
    svc_tuned = LinearSVC(
        C=C_val, class_weight='balanced', max_iter=10000, random_state=SEED
        )
    res = evaluate_cv(clone(svc_tuned), X_train_all, y, f'SVC (C={C_val})')
    results[f'SVC_C{C_val}'] = res

## 4. Model Comparison & Ensemble

In [ ]:
# Summary table
print(f'{"Model":<45} {"CV F1":>10} {"Std":>8} {"OOF F1":>10}')
print('='*75)
sorted_results = sorted(results.items(), key=lambda x: x[1]['mean_f1'], reverse=True)
for name, res in sorted_results:
    print(f'{res["name"]:<45} {res["mean_f1"]:>10.4f} {res["std_f1"]:>8.4f} {res["oof_f1"]:>10.4f}')

best_key = sorted_results[0][0]
print(f'\n Best single model: {results[best_key]["name"]} (CV F1 = {results[best_key]["mean_f1"]:.4f})')

###  Majority Vote Ensemble

Combine the top-N models by unweighted majority voting on their OOF and test predictions.

In [ ]:
# --- Ensemble: Majority Vote from top models ---
# Pick top 5 models by CV F1
top_n = min(5, len(sorted_results))
top_models = sorted_results[:top_n]
print(f'Ensemble of top {top_n} models:')
for name, res in top_models:
    print(f' {res["name"]}: CV F1={res["mean_f1"]:.4f}')

    # OOF ensemble (majority vote)
oof_stack = np.column_stack([res['oof_preds'] for _, res in top_models])
oof_ensemble = np.apply_along_axis(
    lambda x: np.bincount(x.astype(int), minlength=6).argmax(),
    axis=1, arr=oof_stack
    )
ensemble_oof_f1 = f1_score(y, oof_ensemble, average='macro')
print(f'\nEnsemble OOF Macro F1: {ensemble_oof_f1:.4f}')
print('\nPer-class report:')
print(classification_report(y, oof_ensemble,
    target_names=[f'{v}({k})' for k,v in LABEL_MAP.items()]))

# Test ensemble
test_stack = np.column_stack([res['test_preds'] for _, res in top_models])
test_ensemble = np.apply_along_axis(
    lambda x: np.bincount(x.astype(int), minlength=6).argmax(),
    axis=1, arr=test_stack
    )
print(f'\nTest ensemble predictions distribution:')
print(pd.Series(test_ensemble).value_counts().sort_index().to_string())

###  Weighted Majority Voting

Weight each model's vote by its CV macro-F1 score for a stronger ensemble.

In [ ]:
# --- Weighted Ensemble (weight by CV F1) ---
# For each test sample, do weighted voting
weights = np.array([res['mean_f1'] for _, res in top_models])
weights = weights / weights.sum() # normalize

def weighted_vote(preds_matrix, weights, n_classes=6):
    """Weighted majority vote."""
    n_samples = preds_matrix.shape[0]
    result = np.zeros(n_samples, dtype=int)
    for i in range(n_samples):
        class_weights = np.zeros(n_classes)
        for j in range(len(weights)):
            class_weights[int(preds_matrix[i, j])] += weights[j]
            result[i] = class_weights.argmax()
            return result

            # OOF weighted ensemble
oof_weighted = weighted_vote(oof_stack, weights)
weighted_oof_f1 = f1_score(y, oof_weighted, average='macro')
print(f'Weighted Ensemble OOF Macro F1: {weighted_oof_f1:.4f}')

# Test weighted ensemble 
test_weighted = weighted_vote(test_stack, weights)
print(f'\nWeighted test predictions distribution:')
print(pd.Series(test_weighted).value_counts().sort_index().to_string())

###  Weighted Vote Ensemble

Weight each model's vote proportionally to its CV macro-F1 score.

## 5. Confusion Matrix Analysis

###  CV Results Summary

Aggregate and rank all models by macro-F1 score.

###  Weighted Majority Vote

Weighted voting where each model's vote is proportional to its CV macro-F1 score.

In [ ]:
# Confusion matrix for best single model and ensemble
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

best_res = results[best_key]
for ax, preds, title in [
    (axes[0], best_res['oof_preds'], f'Best Single: {best_res["name"]}\nOOF F1={best_res["oof_f1"]:.4f}'),
    (axes[1], oof_ensemble, f'Top-{top_n} Ensemble\nOOF F1={ensemble_oof_f1:.4f}')
    ]:
        cm = confusion_matrix(y, preds)
        cm_pct = cm.astype(float) / cm.sum(axis=1, keepdims=True) * 100
        sns.heatmap(cm_pct, annot=True, fmt='.1f', cmap='Blues', ax=ax,
            xticklabels=[LABEL_MAP[i] for i in range(6)],
            yticklabels=[LABEL_MAP[i] for i in range(6)])
        ax.set_xlabel('Predicted')
        ax.set_ylabel('True')
        ax.set_title(title)

plt.tight_layout()
plt.savefig('figures/classical_confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved to figures/classical_confusion_matrices.png')

## 6. Save Predictions in Multiple Submission Formats
We'll save predictions from the best single model AND the ensemble in **every plausible format** so we can test submission formats efficiently later.

###  Ensemble — Weighted Majority Vote

Combine top-performing models via weighted majority voting, where weights are proportional to each model's CV F1.

In [ ]:
# Decide which predictions to save
# Compare: best single vs ensemble vs weighted ensemble
best_single_f1 = results[best_key]['mean_f1']
print(f'Best single model: {results[best_key]["name"]} → CV F1 = {best_single_f1:.4f}')
print(f'Majority Ensemble: OOF F1 = {ensemble_oof_f1:.4f}')
print(f'Weighted Ensemble: OOF F1 = {weighted_oof_f1:.4f}')
print()

# Pick the best overall approach
candidates = [
    ('best_single', results[best_key]['test_preds'], best_single_f1),
    ('ensemble_vote', test_ensemble, ensemble_oof_f1),
    ('ensemble_weighted', test_weighted, weighted_oof_f1),
    ]
candidates.sort(key=lambda x: x[2], reverse=True)
print('Ranking:')
for name, _, f1 in candidates:
    print(f' {name}: {f1:.4f}')

    # Use the best predictions
best_name_final = candidates[0][0]
best_preds_final = candidates[0][1]
print(f'\n→ Using: {best_name_final}')

###  Save Submission CSV Files

Write predictions in `ID,LABEL` format for competition submission.

In [ ]:
# Save submissions in the required format (ID,LABEL)
os.makedirs('submissions', exist_ok=True)

# Best predictions
lines = ['ID,LABEL'] + [f'{i},{best_preds_final[i]}' for i in range(600)]
with open('submissions/classical_best.csv', 'w') as f:
    f.write('\n'.join(lines) + '\n')
print(f'Saved: submissions/classical_best.csv ({best_name_final})')

# Second-best predictions
second_name = candidates[1][0]
second_preds = candidates[1][1]
lines2 = ['ID,LABEL'] + [f'{i},{second_preds[i]}' for i in range(600)]
with open('submissions/classical_second.csv', 'w') as f:
    f.write('\n'.join(lines2) + '\n')
print(f'Saved: submissions/classical_second.csv ({second_name})')

print(f'\nBest prediction distribution: {dict(zip(*__import__("numpy").unique(best_preds_final, return_counts=True)))}')


###  Phase 2 — Final Summary

Print final summary of all models, best scores, and artifacts.

In [ ]:
# --- Final Summary ---
print('\n' + '='*70)
print('PHASE 2 SUMMARY')
print('='*70)
print(f'\nFeatures: {train_feats.shape[1]} handcrafted + {X_train_word.shape[1]} word TF-IDF + {X_train_char.shape[1]} char TF-IDF')
print(f'Total feature dim: {X_train_all.shape[1]}')
print(f'\nModels trained: {len(results)}')
print(f'\nTop 5 models:')
for i, (name, res) in enumerate(sorted_results[:5]):
    print(f' {i+1}. {res["name"]:40s} CV F1={res["mean_f1"]:.4f} ± {res["std_f1"]:.4f}')
print(f'\nEnsemble OOF F1: {ensemble_oof_f1:.4f}')
print(f'Weighted Ensemble OOF F1: {weighted_oof_f1:.4f}')
print(f'\nBest approach: {best_name_final}')
print(f'\nPrediction files saved in submissions/ -- ready for format testing.')
print('\n Submission format still needs to be resolved before submitting!')

###  Write Submission Files

Write predictions in the required `ID,LABEL` CSV format for competition submission.

In [ ]:
# Save the best predictions from each method
preds = results[best_key]['test_preds']

# Best single model
lines = ['ID,LABEL'] + [f'{i},{results[best_key]["test_preds"][i]}' for i in range(600)]
with open('submissions/classical_best_single.csv', 'w') as f:
    f.write('\n'.join(lines) + '\n')
print(f'Saved: submissions/classical_best_single.csv ({results[best_key]["name"]})')

# Ensemble
lines = ['ID,LABEL'] + [f'{i},{test_ensemble[i]}' for i in range(600)]
with open('submissions/classical_ensemble.csv', 'w') as f:
    f.write('\n'.join(lines) + '\n')
print(f'Saved: submissions/classical_ensemble.csv (majority vote top-{top_n})')

# Weighted ensemble
lines = ['ID,LABEL'] + [f'{i},{test_weighted[i]}' for i in range(600)]
with open('submissions/classical_weighted_ensemble.csv', 'w') as f:
    f.write('\n'.join(lines) + '\n')
print(f'Saved: submissions/classical_weighted_ensemble.csv')

print(f'\nAll submission files use ID,LABEL format.')


###  Persist Models & Vectorizers

Serialize trained classifiers and TF-IDF vectorizers to disk for reuse in ensemble phases.

In [ ]:
# ===== Save trained models & vectorizers for reuse in later notebooks =====
import joblib

os.makedirs('models', exist_ok=True)

# Save TF-IDF vectorizers (fitted on all_texts)
joblib.dump(tfidf_word, 'models/tfidf_word.joblib')
joblib.dump(tfidf_char, 'models/tfidf_char.joblib')
print('Saved: tfidf_word.joblib, tfidf_char.joblib')

# Save feature scaler (fitted on train features)
joblib.dump(scaler, 'models/feature_scaler.joblib')
print('Saved: feature_scaler.joblib')

# Retrain best SVC on full training data and save
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV

best_svc_full = CalibratedClassifierCV(
    LinearSVC(C=5.0, class_weight='balanced', max_iter=10000, random_state=SEED),
    cv=5, method='sigmoid'
    )
best_svc_full.fit(X_train_all, y)
joblib.dump(best_svc_full, 'models/svc_c5_calibrated.joblib')
print('Saved: svc_c5_calibrated.joblib (SVC retrained on full train)')

# Also retrain best LR
best_lr_full = LogisticRegression(
    C=2.0, class_weight='balanced', max_iter=5000, solver='saga', random_state=SEED
    )
best_lr_full.fit(X_train_all, y)
joblib.dump(best_lr_full, 'models/lr_c2.joblib')
print('Saved: lr_c2.joblib (LR retrained on full train)')

# Save OOF predictions & results for later analysis
import json
cv_results = {}
for key, res in sorted_results:
    cv_results[key] = {
        'name': res['name'],
        'mean_f1': float(res['mean_f1']),
        'std_f1': float(res['std_f1']),
        'oof_f1': float(res['oof_f1']),
        }

with open('models/classical_cv_results.json', 'w') as f:
    json.dump(cv_results, f, indent=2)
print('Saved: classical_cv_results.json')

print('\n All models and artifacts saved to models/ for reuse!')